# Team 20: Sheng intent classification
Track A. Starter Kit v2 character TF-IDF + logistic regression baseline.

This notebook performs conservative mechanical preparation and development-only training.
Human review is pending. It does not produce a final accepted contribution file or final test metrics.
Raw text and decisions stay in the git-ignored `private/` directory.


## 1. Environment
In VS Code, open this project folder and select `.venv/Scripts/python.exe` as the kernel.
For Colab, upload the project code and starter-kit folder privately, set the working directory to the project root,
and uncomment the install command below. Upload the raw workbook privately; do not publish it in a notebook.


In [ ]:
# %pip install -r requirements.txt
from pathlib import Path
import pandas as pd
assert Path('prepare_data.py').exists(), 'Open the project root as your working directory.'


## 2. Prepare candidates
Set the workbook path for your machine. The raw workbook is never modified.
NFC and whitespace normalization preserve Sheng spelling, code-switching and punctuation.
Repeated text, explicit non-answers, missing consent, and possible personal information are logged.
Ambiguous meaning is flagged for human review rather than rewritten.
Re-running preparation does not overwrite review decisions.


In [ ]:
from prepare_data import prepare
SOURCE = Path.home() / 'Downloads' / 'Team 20 _ Sheng Form Responses.xlsx'
# Colab example: SOURCE = Path('/content/Team 20 _ Sheng Form Responses.xlsx')
candidates = prepare(SOURCE)
display(pd.crosstab(candidates.provisional_label, candidates.proposed_split))


## 3. Preliminary training
Scenario grouping is declared before modelling: B1+B2 train, B3 development, B4+B5 reserved.
Every answer to one scenario stays together; the vectorizer learns vocabulary/IDF from training only.
The development set is small, labels are provisional, and semantic overlap still needs reviewer checks.
Do not use this score as the final test result or claim these rows are reviewed contributions.


In [ ]:
from train_preliminary import train
result = train()
display(pd.read_csv('artifacts/preliminary/dev_per_label.csv', index_col=0))
predictions = pd.read_csv('artifacts/preliminary/dev_predictions.csv')
display(predictions.loc[~predictions.correct].head(5))


## 4. Human review checkpoint
Send `private/review_decisions.csv` privately to the fluent reviewer. See `REVIEW_GUIDE.md`.
Fill decision, final_label, reviewer_code, reviewer_independent_of_author, reason and any related_group_override.
Do not accept a response because its form column suggests a label: the answer itself must express the intent.
No decision is filled automatically. This notebook stops at preliminary training until review is supplied.

After review: reconcile related groups and duplicates, check the 15/5/5 per-label minimums,
confirm permission and language metadata, then export a genuine reviewed `contribution.csv`.
Run the organiser validator and training scripts without `--practice`.
Freeze model/settings before a single final test evaluation. Do not train on the reserved test examples.


## 5. Local demo
Run this in a terminal at the project root:
```
.\.venv\Scripts\python.exe -m streamlit run team20_app.py
```
The demo loads the actual trained preliminary model, labels its status, and simulates the route.
It always chooses a known class even for unsupported text; probability is uncalibrated.
Local execution is sufficient for development. Public hosting needs a separate release decision.


## 6. Check reviewed data for final export
The final-data pipeline is documented in `FINAL_DATA_WORKFLOW.md`.
Its default command only checks readiness; it does not train or evaluate anything.
Complete the private config with genuine review and permission evidence before exporting.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'finalize_data.py'], check=True)
# Only after readiness passes, run in a terminal:
# python finalize_data.py --export private/final-v1
